In [ ]:
import secrets
print(secrets.token_hex(32))

In [ ]:

!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q


In [ ]:
from google.colab import userdata

names = [
    "JWT_SECRET",
    "NGROK_AUTHTOKEN",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD"
]

for name in names:
    try:
        value = userdata.get(name)
        print(f"{name}: {'Accessible' if value else 'Empty'}")
    except Exception:
        print(f"{name}: Not accessible")

In [ ]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time

DB_NAME = "users.db"
max_login_attempts = 3
lockout_time = 300  # 5 minutes in seconds

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Users table
    c.execute('''CREATE TABLE IF NOT EXISTS users
                 (email TEXT PRIMARY KEY, password BLOB, created_at TEXT)''')

    # Password History table
    c.execute('''CREATE TABLE IF NOT EXISTS password_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  password BLOB,
                  set_at TEXT,
                  FOREIGN KEY(email) REFERENCES users(email))''')

    # Login Attempts table
    c.execute('''CREATE TABLE IF NOT EXISTS login_attempts
                 (email TEXT PRIMARY KEY, attempts INTEGER, last_attempt REAL)''')

    conn.commit()
    conn.close()

    # Ensure Admin Exists
    init_admin()

def init_admin():
    if not check_user_exists("admin@llm.com"):
        register_user("admin@llm.com", "Admin@123")
        print("Admin account created.")

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# --- Rate Limiting ---
def get_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    attempts, _ = get_login_attempts(email)
    new_attempts = attempts + 1
    now = time.time()

    c.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)",
              (email, new_attempts, now))
    conn.commit()
    conn.close()

def reset_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= max_login_attempts:
        if time.time() - last_attempt < lockout_time:
            return True, lockout_time - (time.time() - last_attempt)
        else:
            # Lockout expired
            reset_login_attempts(email)
    return False, 0

# --- User Management ---
def register_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        salt = bcrypt.gensalt()
        hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
        now = _get_timestamp()

        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)", (email, hashed, now))
        c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def authenticate_user(email, password):
    # Check rate limit first in app.py or here? Better to check before auth logic.
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    if data:
        stored_hash = data[0]
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            reset_login_attempts(email) # Success reset
            return True

    # Failed
    increment_login_attempts(email)
    return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
    history = c.fetchall()
    conn.close()

    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            return set_at
    return None

def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode('utf-8'), stored_hash):
            return True
    return False

def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data is not None

def update_password(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(new_password.encode('utf-8'), salt)
    now = _get_timestamp()

    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

    conn.commit()
    conn.close()

# --- Admin Functions ---
def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, created_at FROM users")
    data = c.fetchall()
    conn.close()
    return data

def delete_user(email):
    if email == "admin@llm.com": return False # Protect admin
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM users WHERE email = ?", (email,))
    c.execute("DELETE FROM password_history WHERE email = ?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()
    return True


In [ ]:

%%writefile /content/app.py
import os
import re
import sys
import jwt
import bcrypt
import datetime
import time
import secrets
import smtplib
import streamlit as st

from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --------------------------------------------------
# 1. Page configuration and theme
# --------------------------------------------------
os.makedirs("/content/.streamlit", exist_ok=True)

with open("/content/.streamlit/config.toml", "w") as f:
    f.write(
        '[theme]\n'
        'base="light"\n'
        'primaryColor="#ffd803"\n'
        'backgroundColor="#f9fcfc"\n'
        'secondaryBackgroundColor="#e3f6f5"\n'
        'textColor="#2d334a"\n'
    )

st.set_page_config(
    page_title="OTP Password Reset",
    page_icon="🔐",
    layout="wide"
)

# --------------------------------------------------
# 2. Import the existing database module
# --------------------------------------------------
import sqlite3

MAIN_DB = "/content/infosys_portal.db"


def main_user_exists(email):
    conn = sqlite3.connect(MAIN_DB)
    try:
        row = conn.execute(
            "SELECT 1 FROM users WHERE lower(email) = lower(?)",
            (email.strip(),)
        ).fetchone()
        return row is not None
    finally:
        conn.close()


def init_password_history():
    conn = sqlite3.connect(MAIN_DB)
    try:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_reset_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                password_hash TEXT NOT NULL,
                changed_at TEXT NOT NULL
            )
        """)
        conn.commit()
    finally:
        conn.close()


def main_password_reused(email, new_password):
    conn = sqlite3.connect(MAIN_DB)
    try:
        rows = conn.execute("""
            SELECT password_hash
            FROM password_reset_history
            WHERE lower(email) = lower(?)
        """, (email.strip(),)).fetchall()

        for (stored_hash,) in rows:
            if bcrypt.checkpw(
                new_password.encode("utf-8"),
                stored_hash.encode("utf-8")
            ):
                return True

        # Also check the current password.
        row = conn.execute("""
            SELECT password_hash
            FROM users
            WHERE lower(email) = lower(?)
        """, (email.strip(),)).fetchone()

        if row and bcrypt.checkpw(
            new_password.encode("utf-8"),
            row[0].encode("utf-8")
        ):
            return True

        return False
    finally:
        conn.close()


def update_main_password(email, new_password):
    hashed = bcrypt.hashpw(
        new_password.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")

    conn = sqlite3.connect(MAIN_DB)
    try:
        conn.execute("BEGIN")

        cursor = conn.execute("""
            UPDATE users
            SET password_hash = ?
            WHERE lower(email) = lower(?)
        """, (hashed, email.strip()))

        if cursor.rowcount != 1:
            raise ValueError(
                "Account not found in the main login database."
            )

        conn.execute("""
            INSERT INTO password_reset_history
                (email, password_hash, changed_at)
            VALUES (?, ?, ?)
        """, (
            email.strip().lower(),
            hashed,
            datetime.datetime.now(
                datetime.timezone.utc
            ).isoformat()
        ))

        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


try:
    init_password_history()
except Exception as e:
    st.error(f"Main database initialization failed: {e}")
    st.stop()


# --------------------------------------------------
# 3. Load secrets
# --------------------------------------------------
JWT_SECRET = os.getenv("JWT_SECRET")
SENDER_EMAIL = os.getenv(
    "EMAIL_ADDRESS",
    "nuthisathwika@gmail.com"
)
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
OTP_EXPIRY_MINUTES = 5

if not JWT_SECRET:
    st.error("JWT_SECRET is missing from Colab Secrets.")
    st.stop()

# --------------------------------------------------
# 4. Password hashing helpers for the OTP token
# --------------------------------------------------
def hash_txt(text):
    return bcrypt.hashpw(
        text.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")


def check_txt(text, stored_hash):
    if not text or not stored_hash:
        return False

    try:
        return bcrypt.checkpw(
            text.encode("utf-8"),
            stored_hash.encode("utf-8")
        )
    except (ValueError, TypeError, AttributeError):
        return False

# --------------------------------------------------
# 5. OTP generation and JWT verification
# --------------------------------------------------
def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"


def make_otp_token(email, otp):
    now = datetime.datetime.now(datetime.timezone.utc)

    payload = {
        "sub": email,
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "iat": now,
        "exp": now + datetime.timedelta(
            minutes=OTP_EXPIRY_MINUTES
        )
    }

    return jwt.encode(
        payload,
        JWT_SECRET,
        algorithm="HS256"
    )


def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )

        if (
            payload.get("sub") != email
            or payload.get("type") != "password_reset_otp"
        ):
            return False, "Security token mismatch."

        if check_txt(input_otp, payload.get("otp_hash")):
            return True, "Valid"

        return False, "Invalid 6-digit OTP code."

    except jwt.ExpiredSignatureError:
        return (
            False,
            f"This OTP expired after {OTP_EXPIRY_MINUTES} minutes. "
            "Please request a new one."
        )
    except Exception:
        return False, "Invalid or corrupted verification token."

# --------------------------------------------------
# 6. Send OTP email
# --------------------------------------------------
def send_professional_email(to_email, otp, app_pass):
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Support <{SENDER_EMAIL}>"
    msg["To"] = to_email
    msg["Subject"] = "Infosys Portal - Verification Code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = SENDER_EMAIL

    text_body = (
        f"Your verification code for Infosys Portal is: {otp}\n"
        f"This code expires in {OTP_EXPIRY_MINUTES} minutes.\n"
        "If you did not request this code, ignore this email."
    )

    html_body = f"""
    <!DOCTYPE html>
    <html>
    <body style="font-family:Arial;background:#f9fcfc;padding:20px;">
        <div style="max-width:500px;margin:auto;background:white;
                    border:2px solid #272343;border-radius:12px;
                    padding:30px;text-align:center;">
            <h2 style="color:#272343;">Infosys Portal Verification</h2>
            <p>We received a password-reset request for
               <b>{to_email}</b>.</p>
            <div style="background:#ffd803;color:#272343;
                        font-size:28px;font-weight:bold;
                        letter-spacing:5px;padding:15px;
                        border:2px solid #272343;border-radius:8px;">
                {otp}
            </div>
            <p>This code expires in
               <b>{OTP_EXPIRY_MINUTES} minutes</b>.</p>
            <p style="font-size:12px;color:#718096;">
                If you did not request this code, ignore this email.
            </p>
        </div>
    </body>
    </html>
    """

    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        with smtplib.SMTP(
            "smtp.gmail.com", 587, timeout=20
        ) as server:
            server.starttls()
            server.login(SENDER_EMAIL, app_pass)
            server.sendmail(
                SENDER_EMAIL,
                to_email,
                msg.as_string()
            )

        return True, "Email sent successfully!"

    except Exception as e:
        return False, f"SMTP Error: {e}"

# --------------------------------------------------
# 7. Password validation
# --------------------------------------------------
def valid_password(password):
    return (
        len(password) >= 8
        and re.search(r"[A-Z]", password) is not None
        and re.search(r"[a-z]", password) is not None
        and re.search(r"[0-9]", password) is not None
        and re.search(r"[^A-Za-z0-9]", password) is not None
    )

# --------------------------------------------------
# 8. Session state
# --------------------------------------------------
for key, value in [
    ("stage", "email"),
    ("reset_email", ""),
    ("jwt_token", "")
]:
    if key not in st.session_state:
        st.session_state[key] = value


def restart_flow():
    st.session_state.stage = "email"
    st.session_state.reset_email = ""
    st.session_state.jwt_token = ""
    st.rerun()

# --------------------------------------------------
# 9. Styling
# --------------------------------------------------
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');

html, body, .stApp {
    background: #f9fcfc !important;
    font-family: 'Inter', sans-serif !important;
    color: #2d334a !important;
}
header, footer, #MainMenu {
    visibility: hidden !important;
}
.block-container {
    padding: 3rem 2rem !important;
    max-width: 800px;
}
h1, h2, h3, h4 {
    font-family: 'Poppins', sans-serif !important;
    color: #272343 !important;
}
div[data-testid="stVerticalBlockBorderWrapper"] {
    background: #ffffff !important;
    border: 2px solid #272343 !important;
    border-radius: 16px !important;
    padding: 24px !important;
    box-shadow: 6px 6px 0px #bae8e8 !important;
}
div[data-baseweb="input"] {
    background: #ffffff !important;
    border: 2px solid #272343 !important;
    border-radius: 10px !important;
}
input {
    color: #2d334a !important;
    -webkit-text-fill-color: #2d334a !important;
    font-size: 16px !important;
}
div[data-testid="stButton"] button {
    background: #ffd803 !important;
    color: #272343 !important;
    border: 2px solid #272343 !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    height: 48px !important;
    box-shadow: 4px 4px 0px #272343 !important;
    width: 100%;
}
div[data-testid="stButton"] button:hover {
    background: #e6c300 !important;
}
</style>
""", unsafe_allow_html=True)

# --------------------------------------------------
# 10. Page heading
# --------------------------------------------------
st.markdown("""
<div style="text-align:center;padding:1rem 0 1.5rem;">
    <div style="font-size:48px;margin-bottom:10px;">🔐</div>
    <h1 style="font-size:2.2rem;margin:0;">
        Secure Password Reset
    </h1>
    <p style="color:#64748b;font-size:15px;">
        Verify your identity with a 6-digit OTP code
        (5-minute expiry)
    </p>
</div>
""", unsafe_allow_html=True)

_, col_center, _ = st.columns([1, 2, 1])

with col_center:
    with st.container(border=True):

        # --------------------------------------------------
        # STAGE 1: Email
        # --------------------------------------------------
        if st.session_state.stage == "email":
            st.markdown(
                "<h3 style='text-align:center;'>Enter your email ID</h3>",
                unsafe_allow_html=True
            )

            email = st.text_input(
                "Email Address",
                placeholder="Enter your email ID",
                label_visibility="collapsed"
            ).strip().lower()

            if st.button(
                "Send Verification Code →",
                use_container_width=True
            ):
                if not email:
                    st.error("Please enter an email address.")

                elif not EMAIL_PASSWORD:
                    st.error(
                        "Gmail App Password not found. "
                        "Add EMAIL_PASSWORD to Colab Secrets."
                    )

                else:
                    try:
                        # Use db.py's users.db lookup.
                        if main_user_exists(email):
                            otp = generate_otp()

                            with st.spinner("Sending OTP..."):
                                ok, message = send_professional_email(
                                    email,
                                    otp,
                                    EMAIL_PASSWORD
                                )

                            if ok:
                                st.session_state.reset_email = email
                                st.session_state.jwt_token = make_otp_token(
                                    email,
                                    otp
                                )
                                st.session_state.stage = "otp"
                                st.success(
                                    "OTP sent! Check your email inbox."
                                )
                                time.sleep(1)
                                st.rerun()
                            else:
                                st.error(message)
                        else:
                            st.error(
                                "Email address not registered "
                                "in the login app's database."
                            )

                    except Exception as e:
                        st.error(f"Database error: {e}")

        # --------------------------------------------------
        # STAGE 2: Verify OTP
        # --------------------------------------------------
        elif st.session_state.stage == "otp":
            st.markdown(
                "<h3 style='text-align:center;'>"
                "Enter 6-Digit OTP Code</h3>",
                unsafe_allow_html=True
            )

            st.info(
                f"Code sent to **{st.session_state.reset_email}**. "
                f"Valid for {OTP_EXPIRY_MINUTES} minutes."
            )

            otp_input = st.text_input(
                "6-Digit Verification Code",
                placeholder="Enter your 6-digit code",
                max_chars=6,
                label_visibility="collapsed"
            ).strip()

            c1, c2 = st.columns([1.2, 1])

            if c1.button("Verify Code →", use_container_width=True):
                if len(otp_input) != 6 or not otp_input.isdigit():
                    st.error("Please enter a valid 6-digit OTP.")
                else:
                    ok, message = verify_otp_token(
                        st.session_state.jwt_token,
                        otp_input,
                        st.session_state.reset_email
                    )

                    if ok:
                        st.session_state.stage = "reset"
                        st.success("OTP verified successfully!")
                        time.sleep(1)
                        st.rerun()
                    else:
                        st.error(message)

            if c2.button("← Back", use_container_width=True):
                restart_flow()

        # --------------------------------------------------
        # STAGE 3: Set new password
        # --------------------------------------------------

        # --------------------------------------------------
        # STAGE 3: Set new password
        # --------------------------------------------------
        elif st.session_state.stage == "reset":
            st.markdown(
                "<h3 style='text-align:center;'>"
                "Create New Password</h3>",
                unsafe_allow_html=True
            )

            npw = st.text_input(
                "New Password",
                type="password",
                placeholder="New Password (min 8 chars)"
            )

            confirm_npw = st.text_input(
                "Confirm New Password",
                type="password",
                placeholder="Confirm New Password"
            )

            c1, c2 = st.columns([1.2, 1])

            if c1.button(
                "Update Password →",
                use_container_width=True
            ):
                if not npw or not confirm_npw:
                    st.error("Please fill in both password fields.")

                elif npw != confirm_npw:
                    st.error("Passwords do not match.")

                elif not valid_password(npw):
                    st.error(
                        "Password must contain at least 8 characters, "
                        "one uppercase letter, one lowercase letter, "
                        "one number, and one special character."
                    )

                else:
                    try:
                        reset_email = st.session_state.reset_email

                        if not main_user_exists(reset_email):
                            st.error(
                                "Account not found in the main login database. "
                                "Password was not updated."
                            )

                        elif main_password_reused(reset_email, npw):
                            st.error(
                                "Please choose a password you have not used before."
                            )

                        else:
                            update_main_password(reset_email, npw)

                            st.session_state.jwt_token = ""
                            st.success(
                                "Password updated successfully! "
                                "You can now sign in with your new password."
                            )
                            time.sleep(2)
                            restart_flow()

                    except Exception as e:
                        st.error(f"Password update failed: {e}")

            if c2.button("← Cancel", use_container_width=True):
                restart_flow()


In [ ]:
!pip install pyngrok

In [ ]:
!pip install -q streamlit

In [ ]:
import os

print("Current folder:", os.getcwd())
print("app.py exists:", os.path.exists("app.py"))

In [ ]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Safely retrieve secrets from Colab
try:
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token: ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print(f"⚠️ Secret Error: Make sure EMAIL_PASSWORD and NGROK_AUTHTOKEN exist in Colab Secrets.")

# 🚀 2. Kill ONLY old Streamlit servers (Do NOT kill Ngrok so we don't lock your domain!)
!pkill -f streamlit
time.sleep(2)

# 3. Start Streamlit server cleanly in the background with new UI
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'], env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 🚀 4. Safely Reuse or Connect Tunnel
try:
    tunnels = ngrok.get_tunnels()
    if tunnels:
        public_url = tunnels[0].public_url
        print("♻️ Reusing existing active Ngrok tunnel...")
    else:
        public_url = ngrok.connect(8501).public_url
        print("🚀 Created new Ngrok tunnel...")

    print("=" * 65)
    print(f"👉 OTP Password Reset Portal Live URL: {public_url}")
    print("=" * 65)
    print("⏳ Server running! Press [Ctrl + C] or click the Colab Stop button to shut down.")
    while True: time.sleep(1)

except Exception as e:
    print(f"⚠️ Notice: {e}")
    print("🔄 Waiting 5 seconds for Ngrok cloud servers to release domain lock...")
    ngrok.kill()
    !pkill -f ngrok
    time.sleep(5)  # 🚀 Give Ngrok cloud servers time to release your static endpoint!
    public_url = ngrok.connect(8501).public_url
    print("=" * 65)
    print(f"👉 OTP Password Reset Portal Live URL: {public_url}")
    print("=" * 65)
    while True: time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    !pkill -f streamlit
    print("✅ Streamlit stopped. (Ngrok kept alive for faster re-runs)")


In [ ]:
!pip install streamlit streamlit-option-menu bcrypt PyJWT plotly pyngrok

In [ ]:

%%writefile app.py
import os
import re
import sqlite3
import jwt
import bcrypt
import datetime
import time
import secrets
import smtplib
import streamlit as st
import plotly.graph_objects as go

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu


# ============================================================
# CONFIGURATION
# ============================================================

os.makedirs(".streamlit", exist_ok=True)

with open(".streamlit/config.toml", "w") as f:
    f.write(
        '[theme]\n'
        'base="light"\n'
        'primaryColor="#ffd803"\n'
        'backgroundColor="#f9fcfc"\n'
        'secondaryBackgroundColor="#e3f6f5"\n'
        'textColor="#2d334a"\n'
    )

st.set_page_config(
    page_title="Infosys Portal",
    page_icon="⚡",
    layout="wide"
)

JWT_SECRET = os.getenv(
    "JWT_SECRET",
    "change-this-secret-before-deployment"
)

SENDER_EMAIL = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")

# Separate administrator credentials.
# Change this password before using the application.
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "Admin@123"
ADMIN_EMAIL = "admin@local.invalid"

OTP_EXPIRY_MINUTES = 5

COLORS = {
    "bg_main": "#f9fcfc",
    "bg_sidebar": "#e3f6f5",
    "bg_card": "#ffffff",
    "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a",
    "text_heading": "#272343",
    "text_muted": "#64748b",
    "accent": "#ffd803",
    "accent_hover": "#e6c300",
    "border": "#272343",
    "border_light": "#bae8e8"
}

st.markdown(f"""
<style>
html, body, .stApp {{
    background: {COLORS['bg_main']} !important;
    color: {COLORS['text_main']} !important;
}}
h1, h2, h3 {{
    color: {COLORS['text_heading']} !important;
}}
section[data-testid="stSidebar"] {{
    background: {COLORS['bg_sidebar']} !important;
    border-right: 2px solid {COLORS['border']} !important;
}}
div[data-testid="stButton"] button {{
    background: {COLORS['accent']} !important;
    color: {COLORS['text_heading']} !important;
    border: 2px solid {COLORS['border']} !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    min-height: 44px;
}}
.pn-card {{
    background: white;
    border: 2px solid {COLORS['border']};
    border-radius: 14px;
    padding: 22px;
    box-shadow: 4px 4px 0px {COLORS['border_light']};
}}
</style>
""", unsafe_allow_html=True)


# ============================================================
# DATABASE
# ============================================================

DB_NAME = "infosys_portal.db"


def get_db():
    return sqlite3.connect(DB_NAME, check_same_thread=False)


def hash_txt(value):
    return bcrypt.hashpw(
        value.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")


def check_txt(value, stored_hash):
    if not value or not stored_hash:
        return False

    try:
        return bcrypt.checkpw(
            value.encode("utf-8"),
            stored_hash.encode("utf-8")
        )
    except (ValueError, TypeError):
        return False


with get_db() as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS password_reset_otps (
            email TEXT PRIMARY KEY,
            otp_hash TEXT NOT NULL,
            expires_at REAL NOT NULL,
            attempts INTEGER NOT NULL DEFAULT 0
        )
    """)


# ============================================================
# VALIDATION
# ============================================================

def valid_email(email):
    pattern = (
        r"^[A-Za-z]{2,}[A-Za-z0-9._%+-]*"
        r"@[A-Za-z]{2,}[A-Za-z0-9-]*"
        r"\.[A-Za-z]{2,}$"
    )
    return bool(re.fullmatch(pattern, email))


def valid_password(password):
    return (
        len(password) >= 8
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"\d", password)
        and re.search(r"[^A-Za-z0-9]", password)
    )


PASSWORD_MESSAGE = (
    "Password must have at least 8 characters, "
    "one uppercase letter, one lowercase letter, "
    "one number, and one special character."
)


# ============================================================
# JWT AUTHENTICATION
# ============================================================

def make_jwt(email, role="user", username=""):
    now = datetime.datetime.now(datetime.timezone.utc)

    payload = {
        "email": email,
        "role": role,
        "username": username,
        "iat": now,
        "exp": now + datetime.timedelta(hours=2)
    }

    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try:
        return jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
    except (jwt.InvalidTokenError, TypeError):
        return None


# ============================================================
# EMAIL OTP
# ============================================================

def send_otp_email(email):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, (
            "Email settings are missing. Configure EMAIL_ADDRESS "
            "and EMAIL_PASSWORD in Colab Secrets."
        )

    with get_db() as conn:
        user = conn.execute(
            "SELECT id FROM users WHERE email = ?",
            (email,)
        ).fetchone()

    if not user:
        return False, "No registered account exists for this email."

    otp = f"{secrets.randbelow(1000000):06d}"

    message = MIMEText(
        f"Your Infosys Portal password-reset OTP is: {otp}\n\n"
        f"This OTP expires in {OTP_EXPIRY_MINUTES} minutes.\n"
        "If you did not request this, ignore this email."
    )

    message["Subject"] = "Infosys Portal - Password Reset OTP"
    message["From"] = SENDER_EMAIL
    message["To"] = email

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.starttls()
            server.login(SENDER_EMAIL, EMAIL_PASSWORD)
            server.send_message(message)

        with get_db() as conn:
            conn.execute("""
                INSERT OR REPLACE INTO password_reset_otps
                (email, otp_hash, expires_at, attempts)
                VALUES (?, ?, ?, 0)
            """, (
                email,
                hash_txt(otp),
                time.time() + OTP_EXPIRY_MINUTES * 60
            ))

        return True, "OTP sent to your registered email."

    except Exception:
        return False, (
            "Could not send the email. Check your email address, "
            "Gmail App Password, and SMTP settings."
        )


def verify_otp(email, entered_otp):
    with get_db() as conn:
        row = conn.execute("""
            SELECT otp_hash, expires_at, attempts
            FROM password_reset_otps
            WHERE email = ?
        """, (email,)).fetchone()

        if not row:
            return False, "Request an OTP first."

        otp_hash, expires_at, attempts = row

        if time.time() > expires_at:
            conn.execute(
                "DELETE FROM password_reset_otps WHERE email = ?",
                (email,)
            )
            return False, "OTP expired. Request a new OTP."

        if attempts >= 5:
            conn.execute(
                "DELETE FROM password_reset_otps WHERE email = ?",
                (email,)
            )
            return False, "Too many attempts. Request a new OTP."

        if not check_txt(entered_otp, otp_hash):
            conn.execute("""
                UPDATE password_reset_otps
                SET attempts = attempts + 1
                WHERE email = ?
            """, (email,))
            return False, "Incorrect OTP."

        conn.execute(
            "DELETE FROM password_reset_otps WHERE email = ?",
            (email,)
        )

    return True, "OTP verified successfully."


# ============================================================
# SESSION STATE
# ============================================================

defaults = {
    "token": None,
    "page": "Login",
    "reset_email": None,
    "reset_mode": None,
    "reset_verified": False
}

for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value


def navigate(page):
    st.session_state.page = page
    st.rerun()


def clear_reset():
    st.session_state.reset_email = None
    st.session_state.reset_mode = None
    st.session_state.reset_verified = False


def auth_header(title, subtitle="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:24px 0;">
        <div style="font-size:40px;">⚡</div>
        <h1>Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']}">{subtitle}</p>
        <h3>{title}</h3>
    </div>
    """, unsafe_allow_html=True)


# ============================================================
# LOGIN, SIGNUP, FORGOT PASSWORD, ADMIN LOGIN
# ============================================================

if not st.session_state.token:

    if st.session_state.page not in [
        "Login", "Signup", "Forgot", "Admin Login"
    ]:
        st.session_state.page = "Login"

    _, middle, _ = st.columns([1, 1.5, 1])

    with middle:

        # ----------------------------------------------------
        # LOGIN
        # ----------------------------------------------------
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")

            email = st.text_input(
                "Email address",
                key="login_email"
            ).strip().lower()

            password = st.text_input(
                "Password",
                type="password",
                key="login_password"
            )

            if st.button("Sign In →", use_container_width=True):
                if not email or not password:
                    st.error("Please enter both email and password.")

                elif not valid_email(email):
                    st.error("Enter a valid email address.")

                elif email == "infosys@ai":
                    st.error(
                        "This legacy administrator account cannot sign in "
                        "as a regular user. Use Admin Login."
                    )

                else:
                    with get_db() as conn:
                        user = conn.execute("""
                            SELECT username, password_hash
                            FROM users
                            WHERE email = ?
                        """, (email,)).fetchone()

                    if user and check_txt(password, user[1]):
                        st.session_state.token = make_jwt(
                            email,
                            role="user",
                            username=user[0]
                        )
                        navigate("Dashboard")
                    else:
                        st.error("Invalid email or password.")

            c1, c2 = st.columns(2)

            if c1.button("Create Account", use_container_width=True):
                navigate("Signup")

            if c2.button("Forgot Password", use_container_width=True):
                clear_reset()
                navigate("Forgot")

            if st.button("Admin Login", use_container_width=True):
                navigate("Admin Login")

        # ----------------------------------------------------
        # SIGNUP
        # ----------------------------------------------------
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")

            username = st.text_input("Full name / Username")
            email = st.text_input("Email address").strip().lower()
            password = st.text_input("Password", type="password")
            confirm_password = st.text_input(
                "Confirm password",
                type="password"
            )

            question = st.selectbox(
                "Security Question",
                [
                    "What is your pet name?",
                    "What is your mother's maiden name?",
                    "What is your favourite city?"
                ]
            )

            answer = st.text_input("Your security answer")

            st.caption(PASSWORD_MESSAGE)

            if st.button("Create Account", use_container_width=True):
                if not all([
                    username.strip(),
                    email,
                    password,
                    confirm_password,
                    answer.strip()
                ]):
                    st.error("Please fill in all required fields.")

                elif not valid_email(email):
                    st.error(
                        "Invalid email. Use at least two letters "
                        "before @, after @, and after the final dot."
                    )

                elif not valid_password(password):
                    st.error(PASSWORD_MESSAGE)

                elif password != confirm_password:
                    st.error("Passwords do not match.")

                elif email == "infosys@ai":
                    st.error("This email is reserved.")

                else:
                    try:
                        with get_db() as conn:
                            conn.execute("""
                                INSERT INTO users
                                (username, email, password_hash,
                                 security_question, security_answer_hash)
                                VALUES (?, ?, ?, ?, ?)
                            """, (
                                username.strip(),
                                email,
                                hash_txt(password),
                                question,
                                hash_txt(answer.strip().lower())
                            ))

                        st.success(
                            "Account created successfully. Please sign in."
                        )
                        time.sleep(1)
                        clear_reset()
                        navigate("Login")

                    except sqlite3.IntegrityError:
                        st.error(
                            "That email or username is already registered."
                        )

            if st.button("← Back to Sign In", use_container_width=True):
                navigate("Login")

        # ----------------------------------------------------
        # FORGOT PASSWORD
        # ----------------------------------------------------
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Verify your identity first")

            if not st.session_state.reset_email:
                email = st.text_input(
                    "Registered email address"
                ).strip().lower()

                c1, c2 = st.columns(2)

                if c1.button(
                    "Security Question",
                    use_container_width=True
                ):
                    if not email:
                        st.error("Enter your registered email.")
                    elif not valid_email(email):
                        st.error("Enter a valid email address.")
                    else:
                        with get_db() as conn:
                            user = conn.execute("""
                                SELECT security_question
                                FROM users
                                WHERE email = ?
                            """, (email,)).fetchone()

                        if not user:
                            st.error("No account found for that email.")
                        elif not user[0]:
                            st.error(
                                "No security question is available "
                                "for this account. Try OTP verification."
                            )
                        else:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "sq"
                            st.session_state.reset_verified = False
                            st.rerun()

                if c2.button("Send Email OTP", use_container_width=True):
                    if not email:
                        st.error("Enter your registered email.")
                    elif not valid_email(email):
                        st.error("Enter a valid email address.")
                    else:
                        ok, message = send_otp_email(email)

                        if ok:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.session_state.reset_verified = False
                            st.success(message)
                            st.rerun()
                        else:
                            st.error(message)

            else:
                email = st.session_state.reset_email
                mode = st.session_state.reset_mode

                st.info(f"Account: {email}")

                if not st.session_state.reset_verified:

                    if mode == "sq":
                        with get_db() as conn:
                            row = conn.execute("""
                                SELECT security_question
                                FROM users WHERE email = ?
                            """, (email,)).fetchone()

                        if row:
                            st.write("Security question:")
                            st.info(row[0])

                        answer = st.text_input(
                            "Your answer",
                            key="reset_security_answer"
                        )

                        if st.button("Verify Answer"):
                            with get_db() as conn:
                                row = conn.execute("""
                                    SELECT security_answer_hash
                                    FROM users WHERE email = ?
                                """, (email,)).fetchone()

                            if row and check_txt(
                                answer.strip().lower(), row[0]
                            ):
                                st.session_state.reset_verified = True
                                st.rerun()
                            else:
                                st.error("Incorrect security answer.")

                    elif mode == "otp":
                        otp = st.text_input(
                            "Enter the 6-digit OTP",
                            max_chars=6
                        )

                        if st.button("Verify OTP"):
                            if not re.fullmatch(r"\d{6}", otp):
                                st.error("Enter the 6-digit OTP.")
                            else:
                                ok, message = verify_otp(email, otp)

                                if ok:
                                    st.session_state.reset_verified = True
                                    st.rerun()
                                else:
                                    st.error(message)

                        if st.button("Resend OTP"):
                            ok, message = send_otp_email(email)
                            if ok:
                                st.success(message)
                            else:
                                st.error(message)

                else:
                    st.success("Identity verified. Set a new password.")

                    new_password = st.text_input(
                        "New password",
                        type="password",
                        key="new_password"
                    )

                    confirm_new_password = st.text_input(
                        "Confirm new password",
                        type="password",
                        key="confirm_new_password"
                    )

                    st.caption(PASSWORD_MESSAGE)

                    if st.button("Reset Password"):
                        if not new_password or not confirm_new_password:
                            st.error("Please fill in both password fields.")

                        elif not valid_password(new_password):
                            st.error(PASSWORD_MESSAGE)

                        elif new_password != confirm_new_password:
                            st.error("Passwords do not match.")

                        else:
                            with get_db() as conn:
                                conn.execute("""
                                    UPDATE users
                                    SET password_hash = ?
                                    WHERE email = ?
                                """, (hash_txt(new_password), email))

                                conn.execute("""
                                    DELETE FROM password_reset_otps
                                    WHERE email = ?
                                """, (email,))

                            clear_reset()
                            st.success(
                                "Password reset successfully. Please sign in."
                            )
                            time.sleep(1)
                            navigate("Login")

            if st.button("← Cancel / Back to Login"):
                clear_reset()
                navigate("Login")

        # ----------------------------------------------------
        # SEPARATE ADMIN LOGIN
        # ----------------------------------------------------
        elif st.session_state.page == "Admin Login":
            auth_header("Administrator Login", "Restricted access")

            admin_username = st.text_input("Admin username")
            admin_password = st.text_input(
                "Admin password",
                type="password"
            )

            if st.button("Admin Sign In", use_container_width=True):
                if not admin_username or not admin_password:
                    st.error("Enter both administrator credentials.")

                elif (
                    secrets.compare_digest(
                        admin_username, ADMIN_USERNAME
                    )
                    and secrets.compare_digest(
                        admin_password, ADMIN_PASSWORD
                    )
                ):
                    st.session_state.token = make_jwt(
                        ADMIN_EMAIL,
                        role="admin",
                        username=ADMIN_USERNAME
                    )
                    navigate("Dashboard")

                else:
                    st.error("Invalid administrator credentials.")

            if st.button("← Back to User Login"):
                navigate("Login")


# ============================================================
# DASHBOARD
# ============================================================

else:
    payload = verify_jwt(st.session_state.token)

    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    role = payload.get("role")
    email = payload.get("email")
    username = payload.get("username", "")

    if role == "user":
        with get_db() as conn:
            user = conn.execute("""
                SELECT username, email
                FROM users
                WHERE email = ?
            """, (email,)).fetchone()

        if not user:
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

        username, email = user

    elif role != "admin":
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    with st.sidebar:
        st.markdown("## ⚡ Infosys Portal")
        st.write("Administrator" if role == "admin" else "User Portal")

        if role == "admin":
            selected = option_menu(
                "Admin Menu",
                ["Dashboard", "Registered Users", "Logout"],
                icons=["house", "people", "box-arrow-right"]
            )
        else:
            selected = option_menu(
                "User Menu",
                ["Dashboard", "Analytics", "Reports", "Logout"],
                icons=["house", "graph-up", "file-text", "box-arrow-right"]
            )

    if selected == "Logout":
        st.session_state.token = None
        st.session_state.page = "Login"
        clear_reset()
        st.rerun()

    elif role == "admin":
        st.title("🛡️ Administrator Dashboard")
        st.write(f"Welcome, {ADMIN_USERNAME}.")

        if selected == "Registered Users":
            st.subheader("Registered Users")

            with get_db() as conn:
                users = conn.execute("""
                    SELECT username, email
                    FROM users
                    WHERE email != 'infosys@ai'
                    ORDER BY username
                """).fetchall()

            if users:
                st.dataframe(
                    [
                        {"Username": u[0], "Email": u[1]}
                        for u in users
                    ],
                    use_container_width=True,
                    hide_index=True
                )
            else:
                st.info("No registered users yet.")

        else:
            st.markdown("""
            <div class="pn-card">
                <h3>Admin Control Panel</h3>
                <p>Use Registered Users to view account usernames
                and email addresses.</p>
            </div>
            """, unsafe_allow_html=True)

    else:
        st.title(f"Welcome, {username}!")
        st.write(f"Email: {email}")

        if selected == "Dashboard":
            st.subheader("Dashboard")

            c1, c2, c3, c4 = st.columns(4)

            for col, label, value in [
                (c1, "Documents Indexed", "128"),
                (c2, "Searches Today", "47"),
                (c3, "Efficiency Score", "98.4%"),
                (c4, "Security Status", "Secured")
            ]:
                col.markdown(f"""
                <div class="pn-card">
                    <p>{label}</p>
                    <h2>{value}</h2>
                </div>
                """, unsafe_allow_html=True)

            fig = go.Figure(
                go.Indicator(
                    mode="gauge+number",
                    value=92,
                    title={"text": "System Health Index"},
                    gauge={"axis": {"range": [0, 100]}}
                )
            )

            fig.update_layout(height=300)
            st.plotly_chart(fig, use_container_width=True)

        elif selected == "Analytics":
            st.subheader("Analytics")
            st.info("Analytics section.")

        elif selected == "Reports":
            st.subheader("Reports")
            st.info("Reports section.")


In [ ]:

import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# Load Colab Secrets
for key in ["NGROK_AUTHTOKEN", "EMAIL_ADDRESS",
            "EMAIL_PASSWORD", "JWT_SECRET"]:
    try:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    except Exception:
        pass

ngrok_token = os.environ.get("NGROK_AUTHTOKEN")

if not ngrok_token:
    raise ValueError(
        "NGROK_AUTHTOKEN is missing. Add it in Colab Secrets."
    )

# Stop old processes/tunnels
ngrok.kill()

# Configure ngrok
ngrok.set_auth_token(ngrok_token)

# Start Streamlit
log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for Streamlit to start
for i in range(15):
    time.sleep(1)
    if process.poll() is not None:
        log_file.close()
        print("Streamlit failed to start:")
        with open("/content/streamlit.log") as f:
            print(f.read())
        raise RuntimeError("Check the error above.")

# Create public URL
try:
    tunnel = ngrok.connect(8501, "http")
    print("Infosys Portal URL:", tunnel.public_url)
    print("Open this URL in your browser.")
except Exception as e:
    print("Could not create ngrok tunnel:", e)
    print("\nStreamlit logs:")
    log_file.flush()
    with open("/content/streamlit.log") as f:
        print(f.read())


In [ ]:
!pkill -f "streamlit run"

In [ ]:
!fuser -k 8501/tcp